# 🚀 mpyCUDA — Google Colab Entry Point

**Author**: Schryzon  
**Repository**: [github.com/Schryzon/mpyCUDA](https://github.com/Schryzon/mpyCUDA)

This notebook automatically sets up the environment and runs both parallel workloads:
1. **Matrix Convolution** (`parallel_conv`) — scaling analysis with Amdahl & Gustafson laws
2. **Image Processing** (`parallel_image`) — multiple filters (blur, edge, sobel, sharpen, emboss) with guided MPI scheduling

> ⚠️ **Requires a GPU runtime!** Go to `Runtime → Change runtime type → GPU` before running.

---
## Step 1 — Mount Google Drive & Clone Repository

Your Google Drive will be used to **persist the repository** across sessions.
If the repo already exists in your Drive, it will be reused (not re-cloned).

In [ ]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/Jay-IF24-mpyCUDA'
REPO_URL   = 'https://github.com/Schryzon/mpyCUDA.git'

if not os.path.exists(DRIVE_PATH):
    print('📥 Repository not found in Drive. Cloning now...')
    !git clone "{REPO_URL}" "{DRIVE_PATH}"
    print('✅ Clone complete!')
else:
    print('✅ Repository already exists in Drive. Pulling latest changes...')
    !git -C "{DRIVE_PATH}" pull

# Create a fast symlink in /content so all relative paths work correctly
WORK_DIR = '/content/mpyCUDA'
if not os.path.exists(WORK_DIR):
    !ln -s "{DRIVE_PATH}" "{WORK_DIR}"

print(f'📂 Working directory: {WORK_DIR}')

---
## Step 2 — Install System Dependencies

We install **OpenMPI** and **OpenCV** development headers. This is required every Colab session
(system packages are reset when the runtime restarts). The repo files in Drive are preserved.

In [ ]:
print('📦 Installing OpenMPI and OpenCV dependencies...')
!apt-get install -y -q \
    libopenmpi-dev openmpi-bin \
    libopencv-dev python3-opencv
print('✅ Dependencies installed!')

# Verify tools are available
!echo "--- MPI ---" && mpirun --version
!echo "--- nvcc ---" && nvcc --version
!echo "--- OpenCV ---" && pkg-config --modversion opencv4

---
## Step 3 — Compile CUDA/MPI Programs

Uses the `Makefile` in `scripts/` to compile both programs for the current Linux environment.
The compiled binaries are **not** saved to Drive (they must be recompiled each session).

In [ ]:
SCRIPTS_DIR = f'{WORK_DIR}/Kamen-Rider-Image-Convolution/scripts'

print('🔨 Compiling parallel_conv and parallel_image...')
result = !make -C "{SCRIPTS_DIR}" all 2>&1
print('\n'.join(result))

# Verify binaries exist
for binary in ['parallel_conv', 'parallel_image']:
    path = f'{SCRIPTS_DIR}/{binary}'
    if os.path.exists(path):
        print(f'✅ {binary} compiled successfully')
    else:
        print(f'❌ {binary} FAILED to compile — check output above')

---
# 📊 Part A: Matrix Convolution — Scaling Analysis

This benchmarks `parallel_conv` across multiple processor counts and compares the
actual speedup against **Amdahl's Law** and **Gustafson's Law**.

In [ ]:
import subprocess
import re
import numpy as np
import matplotlib.pyplot as plt

CONV_BIN = f'{SCRIPTS_DIR}/parallel_conv'
procs = [2, 4, 8, 16, 24]
N = 2048  # Matrix dimension
times = []

for p in procs:
    cmd = [
        'mpirun', '--oversubscribe', '--allow-run-as-root',
        '-n', str(p),
        CONV_BIN, str(N)
    ]
    print(f'⏱️  Benchmarking {p} processors (N={N})...')
    res = subprocess.run(cmd, capture_output=True, text=True)

    match = re.search(r'Time taken: ([0-9.]+)', res.stdout)
    if match:
        t = float(match.group(1))
        times.append(t)
        print(f'   → {t:.4f}s')
    else:
        print(f'   ❌ Could not parse time. stdout: {res.stdout[:200]}')
        print(f'      stderr: {res.stderr[:200]}')

In [ ]:
if len(times) == len(procs):
    p_arr = np.array(procs)
    t_arr = np.array(times)
    actual_speedup = t_arr[0] / t_arr

    s = 0.05  # estimated serial fraction
    amdahl   = 1 / (s + (1 - s) / p_arr)
    gustafson = p_arr - s * (p_arr - 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Speedup chart
    axes[0].plot(p_arr, amdahl,        '--o', label="Amdahl's Law (Theoretical)")
    axes[0].plot(p_arr, gustafson,     '--s', label="Gustafson's Law (Theoretical)")
    axes[0].plot(p_arr, actual_speedup, '-^', label='Actual Speedup (Guided MPI+CUDA)', linewidth=2)
    axes[0].set_xlabel('Number of Processors')
    axes[0].set_ylabel('Speedup')
    axes[0].set_title('Speedup Analysis — Matrix Convolution')
    axes[0].legend()
    axes[0].grid(True)

    # Raw time chart
    axes[1].bar([str(p) for p in procs], times, color='steelblue')
    axes[1].set_xlabel('Number of Processors')
    axes[1].set_ylabel('Execution Time (s)')
    axes[1].set_title('Raw Execution Time — Matrix Convolution')
    axes[1].grid(True, axis='y')

    plt.tight_layout()
    plt.savefig(f"{WORK_DIR}/conv_results.png", dpi=150)
    plt.show()
    print('✅ Plot saved to Drive!')
else:
    print('⚠️  Not all benchmarks completed — skipping plot.')

---
# 🖼️ Part B: Image Processing — Advanced Filters

This benchmarks `parallel_image` across multiple processor counts for each image and mode:
**Blur, Basic Edge, Sobel Edge, Sharpen, and Emboss**.

In [ ]:
from pathlib import Path

IMAGE_DIR  = f'{WORK_DIR}/Kamen-Rider-Image-Convolution/images'
IMAGE_BIN  = f'{SCRIPTS_DIR}/parallel_image'
OUTPUT_DIR = f'{IMAGE_DIR}/output'

images      = ['decade.jpg', 'kuuga.jpg', 'w_lunatrigger.png']
modes       = ['blur', 'edge', 'sobel', 'sharpen', 'emboss']
procs_img   = [1, 2, 4, 6, 8, 16, 24, 30, 48]

for mode in modes:
    Path(f'{OUTPUT_DIR}/{mode}').mkdir(parents=True, exist_ok=True)

print(f'Output dirs ready for all modes in: {OUTPUT_DIR}')

In [ ]:
all_times = {mode: [] for mode in modes}

for img_name in images:
    input_img = f'{IMAGE_DIR}/{img_name}'
    stem      = img_name.rsplit('.', 1)[0]

    if not os.path.exists(input_img):
        print(f'⚠️  Input image not found: {input_img} — skipping')
        continue

    for mode in modes:
        times_list = all_times[mode]
        for p in procs_img:
            out_file = f'{OUTPUT_DIR}/{mode}/{stem}_{p}.jpg'
            cmd = [
                'mpirun', '--oversubscribe', '--allow-run-as-root',
                '-n', str(p),
                IMAGE_BIN, input_img, out_file, mode
            ]
            print(f'🖼️  [{mode}] {img_name}  {p} ranks...')
            res = subprocess.run(cmd, capture_output=True, text=True)

            match = re.findall(r'Time taken: ([0-9.]+)', res.stdout)
            if match:
                t = float(match[0])
                times_list.append(t)
                print(f'   → {t:.4f}s')
            else:
                print(f'   ❌ Parse error for {p} ranks.')
                print(f'      stdout: {res.stdout[:300]}')
                print(f'      stderr: {res.stderr[:300]}')

import json
print("Benchmarking complete!")

## Performance Plots
We generate execution time and speedup plots for every processing mode.

In [ ]:
n_images = len(images)
n_procs  = len(procs_img)

for mode in modes:
    times_arr = np.array(all_times[mode], dtype=float).reshape(n_images, n_procs)
    speedup_arr = times_arr[:, [0]] / times_arr
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Execution Time
    for i, img in enumerate(images):
        ax1.plot(procs_img, times_arr[i], marker='o', label=img)
    ax1.set_xlabel('Processors')
    ax1.set_ylabel('Execution Time (s)')
    ax1.set_title(f'Execution Time — {mode.capitalize()}')
    ax1.grid(True)
    ax1.legend()
    
    # Speedup
    for i, img in enumerate(images):
        ax2.plot(procs_img, speedup_arr[i], marker='s', label=img)
    ax2.set_xlabel('Processors')
    ax2.set_ylabel('Speedup (vs 1 proc)')
    ax2.set_title(f'Speedup — {mode.capitalize()}')
    ax2.grid(True)
    ax2.legend()
    
    plt.tight_layout()
    plt.show()